In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
packageVersion("marginaleffects")

[1] ‘0.25.1’

In [4]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "GraduationPaper/RevisetoJournal/9991-MergedData_similarity.csv"))
dim(data)

[1] 317275    103

In [5]:
print(names(data))

  [1] "X"                                           
  [2] "work_id"                                     
  [3] "PublishedYear"                               
  [4] "Facility"                                    
  [5] "num_fac"                                     
  [6] "paper_type"                                  
  [7] "paper_language"                              
  [8] "novel_uzzi"                                  
  [9] "novel_uzzi_bin"                              
 [10] "num_fac_scientist"                           
 [11] "ratio_fac_scientist"                         
 [12] "bin_fac_scientist"                           
 [13] "text_fac_scientist"                          
 [14] "fac_scientist_team"                          
 [15] "num_leader"                                  
 [16] "ratio_leader"                                
 [17] "bin_leader"                                  
 [18] "fac_scientist_lead_relratio"                 
 [19] "fac_scientist_lead_num"                

In [6]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                               PublishedYear 
                                           0 
                                    Facility 
                                           0 
                                     num_fac 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                  novel_uzzi 
                                        2532 
                              novel_uzzi_bin 
                                           0 
                           num_fac_scientist 
                                           0 
                         ratio_fac_scientist 
                                           0 
                           bin_fac_scientist 
                                           0 
                          text_fac_scientist 
                                           0 
                          fac_scientist_team 
                                           0 
                                  num_leader 
                                           0 
                                ratio_leader 
                                           0 
                                  bin_leader 
                                           0 
                 fac_scientist_lead_relratio 
                                           0 
                      fac_scientist_lead_num 
                                           0 
                    fac_scientist_lead_ratio 
                                           0 
                      fac_scientist_lead_bin 
                                           0 
                     fac_scientist_lead_text 
                                           0 
                                      CoType 
                                           0 
                        CoType_Collaboration 
                                           0 
                        CoType_Participation 
                                           0 
                              CoType_Service 
                                           0 
                                lnnum_author 
                                           0 
                                  lnnum_inst 
                                           0 
                               lnnum_country 
                                           0 
                               international 
                                           0 
                             lnnum_reference 
                                           0 
                                 open_access 
                                           0 
                                 RaoStirling 
                                           0 
                                         SDG 
                                           0 
                               lntimescited5 
                                       14695 
                              lntimescited10 
                                       12880 
                             lntimescitedall 
                                           0 
                                 lnab_length 
                                           0 
                           lnmean_career_age 
                                           0 
                       lnex_ld_avg_avgimpact 
                                           0 
                      lnex_ld_avg_insthindex 
                                           0 
                                ex_ld_bin_gs 
                                           0 
                              ex_ld_ratio_gs 
                                           0 
                             ex_ld_bin_sameC 
                                         

In [7]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [8]:
# data <- data %>% filter(!is.na(mean_career_age))
# data <- data %>% filter(!is.na(frac_hype_words))
# data <- data %>% filter(!is.na(source_hindex))
# data <- data %>% filter(!is.na(open_access))
# dim(data)

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$Facility <- as.factor(data$Facility)

In [11]:
data$CoType <- factor(data$CoType)
data <- within(data, CoType <- relevel(CoType, ref = 'Service'))
data$paper_type <- factor(data$paper_type)
data <- within(data, paper_type <- relevel(paper_type, ref = 'review'))
data$text_fac_scientist <- factor(data$text_fac_scientist)
data <- within(data, text_fac_scientist <- relevel(text_fac_scientist, ref = 'NonStaffPart'))
data$fac_scientist_lead_text <- factor(data$fac_scientist_lead_text)
data <- within(data, fac_scientist_lead_text <- relevel(fac_scientist_lead_text, ref = 'NonStaffLead'))
data$open_access <- factor(data$open_access)
data <- within(data, open_access <- relevel(open_access, ref = 'False'))
data$SDG <- factor(data$SDG)
data <- within(data, SDG <- relevel(SDG, ref = 'False'))
data$ex_ld_bin_sameC <- factor(data$ex_ld_bin_sameC)
data <- within(data, ex_ld_bin_sameC <- relevel(ex_ld_bin_sameC, ref = 'NonSame'))
data$ex_ld_bin_gs <- factor(data$ex_ld_bin_gs)
data <- within(data, ex_ld_bin_gs <- relevel(ex_ld_bin_gs, ref = 'GlobalSouth'))
data$ex_ld_max_before_year_with_ih_bin <- factor(data$ex_ld_max_before_year_with_ih_bin)
data <- within(data, ex_ld_max_before_year_with_ih_bin <- relevel(ex_ld_max_before_year_with_ih_bin, ref = 'False'))
data$ex_ld_max_before_year_participation_bin <- factor(data$ex_ld_max_before_year_participation_bin)
data <- within(data, ex_ld_max_before_year_participation_bin <- relevel(ex_ld_max_before_year_participation_bin, ref = 'False'))
data$ex_ld_max_before_year_co_lead_bin <- factor(data$ex_ld_max_before_year_co_lead_bin)
data <- within(data, ex_ld_max_before_year_co_lead_bin <- relevel(ex_ld_max_before_year_co_lead_bin, ref = 'False'))
data$international <- factor(data$international)
data <- within(data, international <- relevel(international, ref = 'domestic'))

In [12]:
paper_level <- "lnnum_author + international + lnnum_reference + num_fac + SDG + lnmean_career_age"
ex_controls <- "lnex_ld_avg_avgimpact + lnex_ld_avg_insthindex + ex_ld_bin_gs + ex_ld_bin_sameC + knowledge_proximity_mean"
moderating <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_with_ih_bin"
moderating2 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_participation_bin"
moderating3 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_co_lead_bin"
disciplines <- "Agricultural.and.Biological.Sciences + Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [13]:
paper_vars <- c("lnnum_author", "international", "lnnum_reference", "num_fac", "SDG", "lnmean_career_age")
ex_vars <- c("lnex_ld_avg_avgimpact", "lnex_ld_avg_insthindex", "ex_ld_bin_gs", "ex_ld_bin_sameC", "knowledge_proximity_mean")
moderating_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_with_ih_bin")
moderating2_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_participation_bin")
moderating3_var <- c("lnex_ld_avg_before_year_prod_fac", "ex_ld_max_before_year_co_lead_bin")
disciplines_vars <- c("Agricultural.and.Biological.Sciences", "Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [14]:
cor_data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0),]
dim(cor_data)

[1] 294879    103

In [17]:
# 假设你要计算相关的变量都在 cor_data 里
# 找出一个包含你所有目标变量的列表
var_list <- c("novel_uzzi_bin", "bin_fac_scientist", "CoType_Collaboration", 
              "CoType_Participation", "CoType_Service", "lnnum_author", 
              "international", "lnnum_reference", "num_fac", "SDG", 
              "lnmean_career_age", "lnex_ld_avg_avgimpact", 
              "lnex_ld_avg_insthindex", "ex_ld_bin_gs", "ex_ld_bin_sameC", 
              "knowledge_proximity_mean", "lnex_ld_avg_before_year_prod_fac", 
              "ex_ld_max_before_year_with_ih_bin")

# 提取子集
cor_subset <- data[, var_list]

# 批量将因子转换为数值 (如果你有普通的字符型，建议先按你的代码转成 factor)
for (col in names(cor_subset)) {
  if (is.factor(cor_subset[[col]])) {
    # 将因子转为数字 (1, 2)，然后减去1变成 (0, 1)
    cor_subset[[col]] <- as.numeric(cor_subset[[col]]) - 1
  } else if (is.character(cor_subset[[col]])) {
    # 如果有遗漏的字符型，强制转为因子再转数值
    cor_subset[[col]] <- as.numeric(as.factor(cor_subset[[col]])) - 1
  }
}

# 检查一下是否全部变成了 numeric
str(cor_subset)

'data.frame':	317275 obs. of  18 variables:
 $ novel_uzzi_bin                   : num  1 1 0 0 0 1 0 1 0 0 ...
 $ bin_fac_scientist                : int  1 0 0 0 0 0 0 0 1 0 ...
 $ CoType_Collaboration             : int  1 0 0 0 0 0 0 0 1 0 ...
 $ CoType_Participation             : int  0 0 0 0 0 0 0 0 0 0 ...
 $ CoType_Service                   : int  0 1 1 1 1 1 1 1 0 1 ...
 $ lnnum_author                     : num  2.2 1.39 1.79 2.2 2.2 ...
 $ international                    : num  1 1 1 1 1 1 0 0 1 0 ...
 $ lnnum_reference                  : num  3.47 3.78 3.14 3.5 3.5 ...
 $ num_fac                          : num  1 1 1 2 2 1 1 1 1 1 ...
 $ SDG                              : num  1 0 0 1 1 0 0 1 1 0 ...
 $ lnmean_career_age                : num  3.59 3.09 3.29 3.1 3.1 ...
 $ lnex_ld_avg_avgimpact            : num  3.12 2.58 4.03 3.67 3.67 ...
 $ lnex_ld_avg_insthindex           : num  5.52 6.04 6.56 6.38 6.38 ...
 $ ex_ld_bin_gs                     : num  1 1 1 1 1 1 0 1 1 1 ...


In [20]:
# 第一步：指定北京大学（PKU）CRAN 镜像源进行安装
# install.packages("Hmisc", repos = "https://mirrors.pku.edu.cn/CRAN/")

# 第二步：安装完成后，加载它
library(Hmisc)

In [21]:
library(Hmisc)
# rcorr 要求输入为矩阵格式
res <- rcorr(as.matrix(cor_subset), type = "spearman")

# 查看相关系数
res$r 

# 查看显著性 P 值
res$P

,novel_uzzi_bin,bin_fac_scientist,CoType_Collaboration,CoType_Participation,CoType_Service,lnnum_author,international,lnnum_reference,num_fac,SDG,lnmean_career_age,lnex_ld_avg_avgimpact,lnex_ld_avg_insthindex,ex_ld_bin_gs,ex_ld_bin_sameC,knowledge_proximity_mean,lnex_ld_avg_before_year_prod_fac,ex_ld_max_before_year_with_ih_bin
novel_uzzi_bin,1.000000000,-0.0083476266,0.004189531,-0.0187786997,0.0083476266,-0.03000795,-0.03233243,0.0179263014,0.001110746,0.051631454,-0.0117026648,-0.02662202,-0.013106378,0.0312148847,-0.001855764,-0.013366574,-0.059692024,-0.0059397973
bin_fac_scientist,-0.008347627,1.0000000000,0.775443622,0.5124924040,-1.0000000000,0.20633084,0.27924434,-0.0009084163,0.192251654,-0.035487987,0.0492260820,-0.01821293,-0.104061133,-0.0072156230,-0.222068693,0.005572081,0.056233408,0.2172964998
CoType_Collaboration,0.004189531,0.7754436220,1.000000000,-0.1447834455,-0.7754436220,0.08067560,0.19297699,-0.0390079428,0.138422930,-0.021691303,0.0562010588,-0.04205353,-0.078637170,0.0069640190,-0.161267214,-0.017202870,0.014551187,0.1496044423
CoType_Participation,-0.018778700,0.5124924040,-0.144783446,1.0000000000,-0.5124924040,0.21361695,0.17515259,0.0516250973,0.113020969,-0.026112578,0.0007095452,0.02864985,-0.056127092,-0.0207779396,-0.128678881,0.032126691,0.068331904,0.1370613249
CoType_Service,0.008347627,-1.0000000000,-0.775443622,-0.5124924040,1.0000000000,-0.20633084,-0.27924434,0.0009084163,-0.192251654,0.035487987,-0.0492260820,0.01821293,0.104061133,0.0072156230,0.222068693,-0.005572081,-0.056233408,-0.2172964998
lnnum_author,-0.030007947,0.2063308354,0.080675599,0.2136169464,-0.2063308354,1.00000000,0.33247460,0.1878182213,0.200258899,0.014997312,0.0409789032,0.31959180,-0.010078194,-0.0377622535,-0.080912316,0.107993135,0.168226363,0.2339379086
international,-0.032332434,0.2792443403,0.192976993,0.1751525930,-0.2792443403,0.33247460,1.00000000,0.0663458816,0.146225978,-0.025142831,0.0439710013,0.06892244,-0.112199246,-0.0990872243,-0.437808260,0.052103814,0.126973882,0.2071208948
lnnum_reference,0.017926301,-0.0009084163,-0.039007943,0.0516250973,0.0009084163,0.18781822,0.06634588,1.0000000000,0.113220925,0.059453561,0.0270984086,0.39878621,0.100125178,-0.0371980313,0.027225492,0.181016372,0.120398399,0.0845620625
num_fac,0.001110746,0.1922516541,0.138422930,0.1130209694,-0.1922516541,0.20025890,0.14622598,0.1132209255,1.000000000,-0.025731499,0.0501545635,0.14054430,0.062378010,0.0481525843,-0.089323033,0.060987583,0.236533707,0.1471648191
SDG,0.051631454,-0.0354879872,-0.021691303,-0.0261125776,0.0354879872,0.01499731,-0.02514283,0.0594535609,-0.025731499,1.000000000,-0.0064453075,0.05293592,0.004308462,-0.0056275199,0.032632160,0.042655098,-0.045417208,-0.0407551246


,novel_uzzi_bin,bin_fac_scientist,CoType_Collaboration,CoType_Participation,CoType_Service,lnnum_author,international,lnnum_reference,num_fac,SDG,lnmean_career_age,lnex_ld_avg_avgimpact,lnex_ld_avg_insthindex,ex_ld_bin_gs,ex_ld_bin_sameC,knowledge_proximity_mean,lnex_ld_avg_before_year_prod_fac,ex_ld_max_before_year_with_ih_bin
novel_uzzi_bin,NA,2.575685e-06,1.828255e-02,0.0000000,2.575685e-06,0.000000e+00,0,0.0000000,0.5315447,0.0000000000,4.339995e-11,0,1.549871e-13,0.000000e+00,2.958861e-01,5.107026e-14,0.000000e+00,0.0008206449
bin_fac_scientist,2.575685e-06,NA,0.000000e+00,0.0000000,0.000000e+00,0.000000e+00,0,0.6088727,0.0000000,0.0000000000,0.000000e+00,0,0.000000e+00,4.815761e-05,0.000000e+00,1.697527e-03,0.000000e+00,0.0000000000
CoType_Collaboration,1.828255e-02,0.000000e+00,NA,0.0000000,0.000000e+00,0.000000e+00,0,0.0000000,0.0000000,0.0000000000,0.000000e+00,0,0.000000e+00,8.757419e-05,0.000000e+00,0.000000e+00,2.220446e-16,0.0000000000
CoType_Participation,0.000000e+00,0.000000e+00,0.000000e+00,NA,0.000000e+00,0.000000e+00,0,0.0000000,0.0000000,0.0000000000,6.894031e-01,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000
CoType_Service,2.575685e-06,0.000000e+00,0.000000e+00,0.0000000,NA,0.000000e+00,0,0.6088727,0.0000000,0.0000000000,0.000000e+00,0,0.000000e+00,4.815761e-05,0.000000e+00,1.697527e-03,0.000000e+00,0.0000000000
lnnum_author,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000,0.000000e+00,NA,0,0.0000000,0.0000000,0.0000000000,0.000000e+00,0,1.371712e-08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000
international,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000,0.000000e+00,0.000000e+00,NA,0.0000000,0.0000000,0.0000000000,0.000000e+00,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000
lnnum_reference,0.000000e+00,6.088727e-01,0.000000e+00,0.0000000,6.088727e-01,0.000000e+00,0,NA,0.0000000,0.0000000000,0.000000e+00,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000
num_fac,5.315447e-01,0.000000e+00,0.000000e+00,0.0000000,0.000000e+00,0.000000e+00,0,0.0000000,NA,0.0000000000,0.000000e+00,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000
SDG,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000,0.000000e+00,0.000000e+00,0,0.0000000,0.0000000,NA,2.828917e-04,0,1.523112e-02,1.525254e-03,0.000000e+00,0.000000e+00,0.000000e+00,0.0000000000


In [ ]:
# 1. 自定义一个函数来计算这5个指标
# 注意：务必加上 na.rm = TRUE，否则只要数据里有一个缺失值(NA)，结果就会变成 NA
get_desc_stats <- function(x) {
  c(Min    = min(x, na.rm = TRUE),
    Max    = max(x, na.rm = TRUE),
    Mean   = mean(x, na.rm = TRUE),
    Median = median(x, na.rm = TRUE),
    Std    = sd(x, na.rm = TRUE))
}
# 2. 将函数批量应用到 cor_subset 的每一列
# sapply 默认返回的是变量在列、指标在行的矩阵，所以我们用 t() 将它转置，让变量变成行
desc_matrix <- t(sapply(cor_subset, get_desc_stats))
# 3. 转成数据框，并统一保留 3 位小数，保持和相关系数矩阵一样整洁
desc_df <- as.data.frame(desc_matrix)
desc_df <- round(desc_df, 3)
# 4. 查看在屏幕上的结果
print(desc_df)
# 5. 导出为 CSV，方便复制进 Word
write.csv(desc_df, file = "descriptive_statistics.csv", row.names = TRUE, fileEncoding = "UTF-8")

print("成功！描述性统计已保存为 descriptive_statistics.csv")

In [22]:
# 1. 提取相关系数和 P 值
r_mat <- res$r
p_mat <- res$P
# 2. 创建一个和 r_mat 同等大小的字符矩阵，用于存放星号
stars_mat <- matrix("", nrow = nrow(r_mat), ncol = ncol(r_mat))
rownames(stars_mat) <- rownames(r_mat)
colnames(stars_mat) <- colnames(r_mat)
# 3. 按照学术规范根据 P 值打星号 (注意：有些变量如果是完全共线性或者对角线，P值可能为 NA)
p_mat[is.na(p_mat)] <- 1  # 把 NA 的 P 值当做 1 处理，避免报错
stars_mat[p_mat < 0.1]  <- "*"
stars_mat[p_mat < 0.05]  <- "**"
stars_mat[p_mat < 0.01] <- "***"
# 4. 把相关系数保留 3 位小数，并和星号拼接在一起
# sprintf("%.3f", r_mat) 保证 0.5 也会显示为 0.500，极其规整
formatted_mat <- matrix(paste0(sprintf("%.3f", r_mat), stars_mat), nrow = nrow(r_mat))
rownames(formatted_mat) <- rownames(r_mat)
colnames(formatted_mat) <- colnames(r_mat)
# 5. 学术界一般只看“下三角”，我们把上三角全部变为空白
formatted_mat[upper.tri(formatted_mat)] <- ""
# 6. 对角线都是自己和自己的相关，强行修改为 "1" 或者留空，这里设为 "1.000"
diag(formatted_mat) <- "1.000"
# 7. 转为数据框 (Data Frame) 准备导出
final_table <- as.data.frame(formatted_mat)
# 8. 导出为 CSV 文件
# 在 Jupyter 中，这个文件会保存在你当前 Notebook 所在的目录下
write.csv(final_table, file = "correlation_matrix_with_stars.csv", row.names = TRUE, fileEncoding = "UTF-8")
print("成功！带星号的相关性矩阵已经保存为 correlation_matrix_with_stars.csv")

[1] "成功！带星号的相关性矩阵已经保存为 correlation_matrix_with_stars.csv"


# H1:With > Without

In [80]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total_bin)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.064958   0.010007   6.491372
lnnum_author                                 -0.087859   0.007299 -12.037309
internationalinternational                   -0.046539   0.009593  -4.851415
lnnum_reference                               0.052699   0.008659   6.085706
num_fac                                       0.066565   0.006729   9.891698
SDGTrue                                       0.084342   0.008152  10.345615
lnmean_career_age                             0.011241   0.013083   0.859195
lnex_ld_avg_avgimpact                        -0.274514   0.007390 -37.145608
lnex_ld_avg_insthindex                       -0.049651   0.007529  -6.594531
ex_ld_bin_gsGlobalNorth                       0.309478   0.017496  17.

In [15]:
# 每组 reg_class 的平均预测概率
pred_bin <- avg_predictions(model_total_bin, variables = "text_fac_scientist")
pred_bin

text_fac_scientist,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonStaffPart,0.3584868,0.03152955,11.36987,5.907745e-30,97.09524,0.2966901,0.4202836
StaffPart,0.3723789,0.03201487,11.63144,2.852572e-31,101.46751,0.3096309,0.4351269


In [16]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_pred.csv")
write.csv(pred_bin, fname, row.names = FALSE)

In [17]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.065$^{***}$\\   
                                                       & (0.010)\\   
   lnnum\_author                                       & -0.088$^{***}$\\   
                                                       & (0.007)\\   
   internationalinternational                          & -0.046$^{***}$\\   
                                                       & (0.010)\\   
   lnnum\_reference                                    & 0.053$^{***}$\\   
                                                       & (0.009)\\   
   num\_fac                                            & 0.067$^{***}$\\   
                                                       & (0.007)\\   
   SDGTru

In [18]:
margins_eff_bin <- avg_comparisons(model_total_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.038752,0.006416741,161.8815,0,Inf,1.026175,1.051328,0.284999,0.2984184,0.284999


In [19]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_comp_ratio.csv")
write.csv(margins_eff_bin, fname, row.names = FALSE)

# H1 different disciplines

In [20]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps_bin)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,575
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.058672   0.010578   5.546590
lnnum_author                                 -0.176638   0.008240 -21.437916
internationalinternational                   -0.075425   0.010529  -7.163785
lnnum_reference                               0.077542   0.009242   8.390263
num_fac                                       0.102222   0.007275  14.051730
SDGTrue                                       0.032981   0.008921   3.696875
lnmean_career_age                            -0.050889   0.014203  -3.583109
lnex_ld_avg_avgimpact                        -0.223156   0.008253 -27.037882
lnex_ld_avg_insthindex                       -0.017083   0.008233  -2.075037
ex_ld_bin_gsGlobalNorth                       0.299839   0.018712  16.

In [21]:
margins_eff_ps_bin <- avg_comparisons(model_ps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.035422,0.006776173,152.8034,0,Inf,1.022141,1.048703,0.2837276,0.295801,0.2837276


In [22]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ps_comp_ratio.csv")
write.csv(margins_eff_ps_bin, fname, row.names = FALSE)

In [23]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.059$^{***}$\\   
                                                       & (0.011)\\   
   lnnum\_author                                       & -0.177$^{***}$\\   
                                                       & (0.008)\\   
   internationalinternational                          & -0.075$^{***}$\\   
                                                       & (0.011)\\   
   lnnum\_reference                                    & 0.077$^{***}$\\   
                                                       & (0.009)\\   
   num\_fac                                            & 0.102$^{***}$\\   
                                                       & (0.007)\\   
   SDGTru

In [24]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls_bin)

NOTE: 1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,590
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
text_fac_scientistStaffPart                   0.131165   0.022359   5.866369
lnnum_author                                  0.232568   0.016875  13.781964
internationalinternational                    0.091356   0.017481   5.225962
lnnum_reference                               0.100861   0.017830   5.656683
num_fac                                      -0.074282   0.012579  -5.904985
SDGTrue                                       0.157392   0.015172  10.373735
lnmean_career_age                             0.149261   0.024350   6.129892
lnex_ld_avg_avgimpact                        -0.386505   0.014080 -27.451053
lnex_ld_avg_insthindex                       -0.136701   0.014241  -9.599456
ex_ld_bin_gsGlobalNorth                       0.056788   0.039531   1.4

In [25]:
margins_eff_ls_bin <- avg_comparisons(model_ls_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_ls_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.065502,0.0144741,73.61439,0,Inf,1.037133,1.09387,0.3164041,0.3454315,0.3164041


In [26]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_ls_comp_ratio.csv")
write.csv(margins_eff_ls_bin, fname, row.names = FALSE)

In [27]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.131$^{***}$\\   
                                                       & (0.022)\\   
   lnnum\_author                                       & 0.233$^{***}$\\   
                                                       & (0.017)\\   
   internationalinternational                          & 0.091$^{***}$\\   
                                                       & (0.018)\\   
   lnnum\_reference                                    & 0.101$^{***}$\\   
                                                       & (0.018)\\   
   num\_fac                                            & -0.074$^{***}$\\   
                                                       & (0.013)\\   
   SDGTrue

In [28]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs_bin)

NOTE: 6 fixed-effects (8 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,409
Fixed-effects: PublishedYear: 40
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    z value
text_fac_scientistStaffPart                    0.388395   0.038336   10.13135
lnnum_author                                   0.319584   0.026505   12.05740
internationalinternational                     0.058681   0.026496    2.21472
lnnum_reference                                0.110130   0.027640    3.98449
num_fac                                       -0.123160   0.020739   -5.93845
SDGTrue                                        0.224109   0.023466    9.55045
lnmean_career_age                              0.184938   0.038681    4.78107
lnex_ld_avg_avgimpact                         -0.334968   0.022019  -15.21267
lnex_ld_avg_insthindex                        -0.120245   0.021503   -5.59208
ex_ld_bin_gsGlobalNorth                        0.232135   0.0

In [29]:
margins_eff_hs_bin <- avg_comparisons(model_hs_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_hs_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.16192,0.04168972,27.87066,6.052872e-171,565.4521,1.08021,1.24363,0.7082483,0.7816464,0.7082483


In [30]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_hs_comp_ratio.csv")
write.csv(margins_eff_hs_bin, fname, row.names = FALSE)

In [31]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.388$^{***}$\\   
                                                       & (0.038)\\   
   lnnum\_author                                       & 0.320$^{***}$\\   
                                                       & (0.026)\\   
   internationalinternational                          & 0.059$^{**}$\\   
                                                       & (0.026)\\   
   lnnum\_reference                                    & 0.110$^{***}$\\   
                                                       & (0.028)\\   
   num\_fac                                            & -0.123$^{***}$\\   
                                                       & (0.021)\\   
   SDGTrue 

In [32]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps_bin <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps_bin)

NOTE: 6 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,294
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
text_fac_scientistStaffPart                    0.317739   0.036403    8.728335
lnnum_author                                   0.344890   0.023766   14.512119
internationalinternational                     0.107954   0.024142    4.471666
lnnum_reference                               -0.088392   0.026780   -3.300651
num_fac                                       -0.131859   0.018676   -7.060333
SDGTrue                                        0.260102   0.021372   12.169968
lnmean_career_age                              0.266285   0.035370    7.528505
lnex_ld_avg_avgimpact                         -0.534767   0.021066  -25.385094
lnex_ld_avg_insthindex                        -0.190767   0.020023   -9.527312
ex_ld_bin_gsGlobalNorth                        0.15

In [33]:
margins_eff_nps_bin <- avg_comparisons(model_nps_bin, variables = "text_fac_scientist", comparison = 'ratio')
margins_eff_nps_bin

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
text_fac_scientist,mean(StaffPart) / mean(NonStaffPart),1.166542,0.03774749,30.90384,1.060737e-209,694.1979,1.092559,1.240526,0.6709483,0.7369578,0.6709483


In [34]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h1_nps_comp_ratio.csv")
write.csv(margins_eff_nps_bin, fname, row.names = FALSE)

In [35]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps_bin,
                           keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   text\_fac\_scientistStaffPart                       & 0.318$^{***}$\\   
                                                       & (0.036)\\   
   lnnum\_author                                       & 0.345$^{***}$\\   
                                                       & (0.024)\\   
   internationalinternational                          & 0.108$^{***}$\\   
                                                       & (0.024)\\   
   lnnum\_reference                                    & -0.088$^{***}$\\   
                                                       & (0.027)\\   
   num\_fac                                            & -0.132$^{***}$\\   
                                                       & (0.019)\\   
   SDGTru

# H2: Collaboration > Participation

In [36]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_total <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_total)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.093664   0.011280   8.303659
CoTypeParticipation                           0.001874   0.015079   0.124291
lnnum_author                                 -0.082752   0.007391 -11.196746
internationalinternational                   -0.046879   0.009593  -4.886663
lnnum_reference                               0.053317   0.008660   6.156694
num_fac                                       0.066123   0.006732   9.822587
SDGTrue                                       0.083797   0.008154  10.276936
lnmean_career_age                             0.008714   0.013093   0.665580
lnex_ld_avg_avgimpact                        -0.275466   0.007394 -37.255718
lnex_ld_avg_insthindex                       -0.049325   0.007528  -6.

In [37]:
# 每组 reg_class 的平均预测概率
pred <- avg_predictions(model_total, variables = "CoType")
pred

CoType,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Service,0.3585577,0.03153474,11.37024,5.882342e-30,97.10145,0.2967507,0.4203646
Collaboration,0.3786608,0.03224739,11.74237,7.728990e-32,103.35142,0.3154571,0.4418646
Participation,0.3589555,0.03162498,11.35038,7.384392e-30,96.77336,0.2969716,0.4209393


In [38]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_pred.csv")
write.csv(pred, fname, row.names = FALSE)

In [39]:
# 每组 reg_class 的平均预测概率
margins_eff <- avg_comparisons(model_total, variables = "CoType", comparison = 'ratio')
margins_eff

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.056067,0.007531083,140.2277,0,Inf,1.0413061,1.070827,0.2846939,0.3041460,0.2846939
CoType,mean(Participation) / mean(Service),1.001109,0.008929571,112.1117,0,Inf,0.9836078,1.018611,0.2846939,0.2850758,0.2846939


In [40]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_comp_ratio.csv")
write.csv(margins_eff, fname, row.names = FALSE)

In [41]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.094$^{***}$\\   
                                                       & (0.011)\\   
   CoTypeParticipation                                 & 0.002\\   
                                                       & (0.015)\\   
   lnnum\_author                                       & -0.083$^{***}$\\   
                                                       & (0.007)\\   
   internationalinternational                          & -0.047$^{***}$\\   
                                                       & (0.010)\\   
   lnnum\_reference                                    & 0.053$^{***}$\\   
                                                       & (0.009)\\   
   num\_fac      

# H2 Discipline

In [42]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ps)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,575
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.081750   0.011854   6.896493
CoTypeParticipation                           0.006896   0.015894   0.433865
lnnum_author                                 -0.172038   0.008338 -20.633222
internationalinternational                   -0.075657   0.010529  -7.185651
lnnum_reference                               0.078010   0.009242   8.440694
num_fac                                       0.101840   0.007277  13.994286
SDGTrue                                       0.032604   0.008922   3.654131
lnmean_career_age                            -0.052982   0.014213  -3.727710
lnex_ld_avg_avgimpact                        -0.224111   0.008258 -27.139673
lnex_ld_avg_insthindex                       -0.016826   0.008231  -2.

In [43]:
# 每组 reg_class 的平均预测概率
margins_eff_ps <- avg_comparisons(model_ps, variables = "CoType", comparison = 'ratio')
margins_eff_ps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.049506,0.007822176,134.1706,0,Inf,1.034175,1.064837,0.2834341,0.3003272,0.2834341
CoType,mean(Participation) / mean(Service),1.004135,0.009545858,105.1906,0,Inf,0.985425,1.022844,0.2834341,0.2848368,0.2834341


In [44]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ps_comp_ratio.csv")
write.csv(margins_eff_ps, fname, row.names = FALSE)

In [45]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.082$^{***}$\\   
                                                       & (0.012)\\   
   CoTypeParticipation                                 & 0.007\\   
                                                       & (0.016)\\   
   lnnum\_author                                       & -0.172$^{***}$\\   
                                                       & (0.008)\\   
   internationalinternational                          & -0.076$^{***}$\\   
                                                       & (0.011)\\   
   lnnum\_reference                                    & 0.078$^{***}$\\   
                                                       & (0.009)\\   
   num\_fac      

In [46]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ls <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_ls)

NOTE: 1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,590
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.112862   0.025274   4.465483
CoTypeParticipation                           0.177887   0.037424   4.753252
lnnum_author                                  0.232349   0.016872  13.770985
internationalinternational                    0.091095   0.017482   5.210651
lnnum_reference                               0.100646   0.017834   5.643612
num_fac                                      -0.074019   0.012581  -5.883650
SDGTrue                                       0.157520   0.015173  10.381696
lnmean_career_age                             0.150727   0.024366   6.185879
lnex_ld_avg_avgimpact                        -0.386252   0.014079 -27.434283
lnex_ld_avg_insthindex                       -0.136568   0.014240  -9.5

In [47]:
# 每组 reg_class 的平均预测概率
margins_eff_ls <- avg_comparisons(model_ls, variables = "CoType", comparison = 'ratio')
margins_eff_ls

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.056350,0.01503416,70.26328,0,Inf,1.026883,1.085816,0.316494,0.3413983,0.316494
CoType,mean(Participation) / mean(Service),1.088891,0.02220913,49.02899,0,Inf,1.045362,1.132420,0.316494,0.3561660,0.316494


In [48]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_ls_comp_ratio.csv")
write.csv(margins_eff_ls, fname, row.names = FALSE)

In [49]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_ls,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.113$^{***}$\\   
                                                       & (0.025)\\   
   CoTypeParticipation                                 & 0.178$^{***}$\\   
                                                       & (0.037)\\   
   lnnum\_author                                       & 0.232$^{***}$\\   
                                                       & (0.017)\\   
   internationalinternational                          & 0.091$^{***}$\\   
                                                       & (0.018)\\   
   lnnum\_reference                                    & 0.101$^{***}$\\   
                                                       & (0.018)\\   
   num\_fac

In [50]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_hs <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
summary(model_hs)

NOTE: 6 fixed-effects (8 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,409
Fixed-effects: PublishedYear: 40
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    z value
CoTypeCollaboration                            0.347392   0.044522    7.80271
CoTypeParticipation                            0.478942   0.063318    7.56410
lnnum_author                                   0.318061   0.026497   12.00377
internationalinternational                     0.058039   0.026499    2.19023
lnnum_reference                                0.110106   0.027655    3.98134
num_fac                                       -0.123250   0.020755   -5.93839
SDGTrue                                        0.224452   0.023469    9.56391
lnmean_career_age                              0.187322   0.038706    4.83968
lnex_ld_avg_avgimpact                         -0.334405   0.022013  -15.19152
lnex_ld_avg_insthindex                        -0.120006   0.0

In [51]:
# 每组 reg_class 的平均预测概率
margins_eff_hs <- avg_comparisons(model_hs, variables = "CoType", comparison = 'ratio')
margins_eff_hs

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.145328,0.03903589,29.34038,3.169837e-189,626.1800,1.068819,1.221837,0.708386,0.7746839,0.708386
CoType,mean(Participation) / mean(Service),1.198086,0.05429612,22.06578,6.739312e-108,356.0156,1.091668,1.304505,0.708386,0.7968141,0.708386


In [52]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_hs_comp_ratio.csv")
write.csv(margins_eff_hs, fname, row.names = FALSE)

In [53]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_hs,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.347$^{***}$\\   
                                                       & (0.044)\\   
   CoTypeParticipation                                 & 0.479$^{***}$\\   
                                                       & (0.063)\\   
   lnnum\_author                                       & 0.318$^{***}$\\   
                                                       & (0.026)\\   
   internationalinternational                          & 0.058$^{**}$\\   
                                                       & (0.026)\\   
   lnnum\_reference                                    & 0.110$^{***}$\\   
                                                       & (0.028)\\   
   num\_fac 

In [54]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_nps <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
summary(model_nps)

NOTE: 6 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,294
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error     z value
CoTypeCollaboration                            0.281504   0.042795    6.578026
CoTypeParticipation                            0.395612   0.059091    6.694940
lnnum_author                                   0.344441   0.023759   14.497001
internationalinternational                     0.107923   0.024142    4.470310
lnnum_reference                               -0.088769   0.026784   -3.314210
num_fac                                       -0.131631   0.018687   -7.044027
SDGTrue                                        0.260535   0.021376   12.188100
lnmean_career_age                              0.268377   0.035399    7.581545
lnex_ld_avg_avgimpact                         -0.534705   0.021070  -25.378090
lnex_ld_avg_insthindex                        -0.19

In [55]:
# 每组 reg_class 的平均预测概率
margins_eff_nps <- avg_comparisons(model_nps, variables = "CoType", comparison = 'ratio')
margins_eff_nps

term,contrast,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),1.147427,0.03664651,31.31067,3.339641e-215,712.4748,1.075601,1.219253,0.6711659,0.7300680,0.6711659
CoType,mean(Participation) / mean(Service),1.207672,0.05131421,23.53484,1.794850e-122,404.4314,1.107098,1.308246,0.6711659,0.7519566,0.6711659


In [56]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h2_nps_comp_ratio.csv")
write.csv(margins_eff_nps, fname, row.names = FALSE)

In [57]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_nps,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                 & 0.282$^{***}$\\   
                                                       & (0.043)\\   
   CoTypeParticipation                                 & 0.396$^{***}$\\   
                                                       & (0.059)\\   
   lnnum\_author                                       & 0.344$^{***}$\\   
                                                       & (0.024)\\   
   internationalinternational                          & 0.108$^{***}$\\   
                                                       & (0.024)\\   
   lnnum\_reference                                    & -0.089$^{***}$\\   
                                                       & (0.027)\\   
   num\_fa

# H3: Too much will suppress

# H3a: Participation too much not good

In [58]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ ratio_fac_scientist + I(ratio_fac_scientist^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_pratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_pratio)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
ratio_fac_scientist                           0.399062   0.071372   5.591267
I(ratio_fac_scientist^2)                     -0.433564   0.122480  -3.539884
lnnum_author                                 -0.082493   0.007276 -11.337694
internationalinternational                   -0.046839   0.009596  -4.881206
lnnum_reference                               0.053172   0.008663   6.137694
num_fac                                       0.066116   0.006744   9.804320
SDGTrue                                       0.084209   0.008153  10.328778
lnmean_career_age                             0.010584   0.013087   0.808759
lnex_ld_avg_avgimpact                        -0.274666   0.007390 -37.164694
lnex_ld_avg_insthindex                       -0.049450   0.007532  -6.

In [59]:
library(marginaleffects)
# 设置 draw = FALSE，直接拦截绘图数据
plot_data <- plot_predictions(model_h3_pratio, condition = "ratio_fac_scientist", draw = FALSE)
# 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
export_data <- plot_data[, c("ratio_fac_scientist", "estimate", "conf.low", "conf.high")]
# # 导出为 CSV 文件，给 Python 准备
write.csv(export_data, "R_ex_ld_h3_pred_pratio.csv", row.names = FALSE)
export_data
# print("数据已成功导出！")
# head(export_data)

ratio_fac_scientist,estimate,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>
0.00000000,0.2482029,0.1974836,0.3069658
0.02003023,0.2496648,0.1987348,0.3086196
0.04006047,0.2510671,0.1999325,0.3102091
0.06009070,0.2524089,0.2010766,0.3117325
0.08012094,0.2536896,0.2021670,0.3131885
0.10015117,0.2549085,0.2032036,0.3145756
0.12018141,0.2560650,0.2041863,0.3158927
0.14021164,0.2571584,0.2051149,0.3171384
0.16024187,0.2581883,0.2059893,0.3183119


In [60]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_pratio,
                           keep = c("ratio_fac_scientist", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   ratio\_fac\_scientist                               & 0.399$^{***}$\\   
                                                       & (0.071)\\   
   ratio\_fac\_scientist square                        & -0.434$^{***}$\\   
                                                       & (0.122)\\   
   lnnum\_author                                       & -0.083$^{***}$\\   
                                                       & (0.007)\\   
   internationalinternational                          & -0.047$^{***}$\\   
                                                       & (0.010)\\   
   lnnum\_reference                                    & 0.053$^{***}$\\   
                                                       & (0.009)\\   
   num\_

# H3b: Lead too much not good

In [61]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ fac_scientist_lead_ratio + I(fac_scientist_lead_ratio^2) +  ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_h3_lratio <- feglm(fml, data = data[(data$paper_type =='article')&(data$CoType_Service==0)&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_h3_lratio)

NOTE: 1 fixed-effect (11 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 78,522
Fixed-effects: PublishedYear: 50
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
fac_scientist_lead_ratio                      0.254071   0.107512   2.363190
I(fac_scientist_lead_ratio^2)                -0.397031   0.159333  -2.491822
lnnum_author                                 -0.286696   0.015260 -18.787212
internationalinternational                   -0.095836   0.022500  -4.259467
lnnum_reference                               0.099232   0.017992   5.515410
num_fac                                       0.105508   0.010548  10.002539
SDGTrue                                       0.086080   0.016575   5.193481
lnmean_career_age                            -0.138351   0.030250  -4.573525
lnex_ld_avg_avgimpact                        -0.188596   0.014787 -12.754104
lnex_ld_avg_insthindex                       -0.043868   0.014598  -3.0

In [62]:
library(marginaleffects)
# 设置 draw = FALSE，直接拦截绘图数据
plot_data <- plot_predictions(model_h3_lratio, condition = "fac_scientist_lead_ratio", draw = FALSE)
# 选出我们最需要的几列：x轴变量、预测值(estimate)、置信区间下限(conf.low)、上限(conf.high)
export_data <- plot_data[, c("fac_scientist_lead_ratio", "estimate", "conf.low", "conf.high")]
# # 导出为 CSV 文件，给 Python 准备
write.csv(export_data, "R_ex_ld_h3_pred_lratio.csv", row.names = FALSE)
export_data
# print("数据已成功导出！")
# head(export_data)

fac_scientist_lead_ratio,estimate,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>
0.00000000,0.2282580,0.1399389,0.3496565
0.01992225,0.2291231,0.1405339,0.3507653
0.03984451,0.2299346,0.1410897,0.3518100
0.05976676,0.2306922,0.1416063,0.3527891
0.07968902,0.2313955,0.1420839,0.3537015
0.09961127,0.2320441,0.1425226,0.3545462
0.11953353,0.2326377,0.1429224,0.3553220
0.13945578,0.2331761,0.1432835,0.3560282
0.15937804,0.2336588,0.1436060,0.3566637


In [63]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_h3_lratio,
                           keep = c("fac_scientist_lead_ratio", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                 & novel\_uzzi\_bin\\    
   Model:                                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_scientist\_lead\_ratio                         & 0.254$^{**}$\\   
                                                       & (0.108)\\   
   fac\_scientist\_lead\_ratio square                  & -0.397$^{**}$\\   
                                                       & (0.159)\\   
   lnnum\_author                                       & -0.287$^{***}$\\   
                                                       & (0.015)\\   
   internationalinternational                          & -0.096$^{***}$\\   
                                                       & (0.022)\\   
   lnnum\_reference                                    & 0.099$^{***}$\\   
                                                       & (0.018)\\   
   num\_fa

# Moderating

In [14]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnex_ld_avg_before_year_prod_fac  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                      Estimate Std. Error
CoTypeCollaboration                                   0.077524   0.025548
CoTypeParticipation                                   0.183022   0.036288
lnex_ld_avg_before_year_prod_fac                     -0.034970   0.005794
lnnum_author                                         -0.077789   0.007564
internationalinternational                           -0.048093   0.009603
lnnum_reference                                       0.053375   0.008660
num_fac                                               0.067123   0.006768
SDGTrue                                               0.083389   0.008155
lnmean_career_age                                     0.007946   0.013100
lnex_ld_avg_avgimpact                                -0.275959   0.007399
lnex_ld_avg_insthindex    

In [15]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnex_ld_avg_before_year_with_ih  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                     Estimate Std. Error
CoTypeCollaboration                                  0.189021   0.019932
CoTypeParticipation                                  0.294749   0.027377
lnex_ld_avg_before_year_with_ih                      0.143061   0.008074
lnnum_author                                        -0.070425   0.007635
internationalinternational                          -0.052485   0.009604
lnnum_reference                                      0.055130   0.008667
num_fac                                              0.068463   0.006746
SDGTrue                                              0.085719   0.008162
lnmean_career_age                                    0.004793   0.013110
lnex_ld_avg_avgimpact                               -0.276463   0.007407
lnex_ld_avg_insthindex               

In [41]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnex_ld_avg_before_year_participation  + ", paper_level, "+", ex_controls, "+ lnex_ld_avg_before_year_prod_fac ",  "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                           Estimate Std. Error
CoTypeCollaboration                                        0.177073   0.014540
CoTypeParticipation                                        0.319217   0.021682
lnex_ld_avg_before_year_participation                      0.120695   0.011073
lnnum_author                                              -0.046444   0.007749
internationalinternational                                -0.044135   0.009611
lnnum_reference                                            0.057652   0.008641
num_fac                                                    0.061298   0.006750
SDGTrue                                                    0.083007   0.008152
lnmean_career_age                                          0.019726   0.013073
lnex_ld_avg_avgimpact                             

In [42]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", "lnex_ld_avg_before_year_participation"),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                           & novel\_uzzi\_bin\\    
   Model:                                                                        & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                           & 0.177$^{***}$\\   
                                                                                 & (0.015)\\   
   CoTypeParticipation                                                           & 0.319$^{***}$\\   
                                                                                 & (0.022)\\   
   lnex\_ld\_avg\_before\_year\_participation                                    & 0.121$^{***}$\\   
                                                                                 & (0.011)\\   
   CoTypeCollaboration $\times$ lnex\_ld\_avg\_before\_year\_participation       & -0.115$^{***

In [36]:
# 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
min_val <- min(data$lnex_ld_avg_before_year_participation, na.rm=TRUE)
max_val <- max(data$lnex_ld_avg_before_year_participation, na.rm=TRUE)
# 2. 运行估计
res_pre_facpub <- avg_comparisons(
    model_pre_facpub,
    variables = "CoType",
    comparison = "ratio",
    newdata = datagrid(
    model = model_pre_facpub,
    lnex_ld_avg_before_year_participation = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
  ),
    by = "lnex_ld_avg_before_year_participation"
)
res_pre_facpub

rowid,term,contrast,lnex_ld_avg_before_year_participation,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,CoType,mean(Collaboration) / mean(Service),0.0000000,1.1432238,0.013665503,83.65765,0.000000e+00,Inf,1.1164399,1.1700077,0.2280035,0.2606591,0.2280035
2,CoType,mean(Collaboration) / mean(Service),0.1318028,1.1299023,0.012373630,91.31535,0.000000e+00,Inf,1.1056504,1.1541541,0.2308157,0.2607992,0.2308157
3,CoType,mean(Collaboration) / mean(Service),0.2636056,1.1167860,0.011300257,98.82838,0.000000e+00,Inf,1.0946379,1.1389341,0.2336521,0.2609394,0.2336521
4,CoType,mean(Collaboration) / mean(Service),0.3954085,1.1038719,0.010485475,105.27629,0.000000e+00,Inf,1.0833208,1.1244231,0.2365126,0.2610796,0.2365126
5,CoType,mean(Collaboration) / mean(Service),0.5272113,1.0911569,0.009965852,109.48958,0.000000e+00,Inf,1.0716242,1.1106896,0.2393972,0.2612199,0.2393972
6,CoType,mean(Collaboration) / mean(Service),0.6590141,1.0786380,0.009762539,110.48744,0.000000e+00,Inf,1.0595038,1.0977722,0.2423058,0.2613603,0.2423058
7,CoType,mean(Collaboration) / mean(Service),0.7908169,1.0663121,0.009870804,108.02688,0.000000e+00,Inf,1.0469657,1.0856585,0.2452384,0.2615007,0.2452384
8,CoType,mean(Collaboration) / mean(Service),0.9226198,1.0541763,0.010258757,102.75868,0.000000e+00,Inf,1.0340695,1.0742831,0.2481948,0.2616411,0.2481948
9,CoType,mean(Collaboration) / mean(Service),1.0544226,1.0422278,0.010876922,95.82010,0.000000e+00,Inf,1.0209094,1.0635461,0.2511750,0.2617816,0.2511750


In [37]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4c_comp_pre_participation.csv")
write.csv(res_pre_facpub, fname, row.names = FALSE)

In [43]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*lnex_ld_avg_before_year_co_lead  + ", paper_level, "+", ex_controls, "+ lnex_ld_avg_before_year_prod_fac ", "+",disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                     Estimate Std. Error
CoTypeCollaboration                                  0.241026   0.018547
CoTypeParticipation                                  0.194909   0.024565
lnex_ld_avg_before_year_co_lead                      0.236184   0.007843
lnnum_author                                        -0.055695   0.007407
internationalinternational                          -0.043925   0.009593
lnnum_reference                                      0.057563   0.008659
num_fac                                              0.063569   0.006738
SDGTrue                                              0.085222   0.008156
lnmean_career_age                                    0.014247   0.013083
lnex_ld_avg_avgimpact                               -0.276988   0.007398
lnex_ld_avg_insthindex               

In [44]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", "lnex_ld_avg_before_year_co_lead"),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                       & novel\_uzzi\_bin\\    
   Model:                                                                    & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                       & 0.241$^{***}$\\   
                                                                             & (0.018)\\   
   CoTypeParticipation                                                       & 0.195$^{***}$\\   
                                                                             & (0.025)\\   
   lnex\_ld\_avg\_before\_year\_co\_lead                                     & 0.236$^{***}$\\   
                                                                             & (0.008)\\   
   CoTypeCollaboration $\times$ lnex\_ld\_avg\_before\_year\_co\_lead        & -0.183$^{***}$\\   
                            

In [39]:
# 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
min_val <- min(data$lnex_ld_avg_before_year_co_lead, na.rm=TRUE)
max_val <- max(data$lnex_ld_avg_before_year_co_lead, na.rm=TRUE)
# 2. 运行估计
res_pre_facpub <- avg_comparisons(
    model_pre_facpub,
    variables = "CoType",
    comparison = "ratio",
    newdata = datagrid(
    model = model_pre_facpub,
    lnex_ld_avg_before_year_co_lead = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
  ),
    by = "lnex_ld_avg_before_year_co_lead"
)
res_pre_facpub

rowid,term,contrast,lnex_ld_avg_before_year_co_lead,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,CoType,mean(Collaboration) / mean(Service),0.0000000,1.2047486,0.018588781,64.81053,0,Inf,1.1683153,1.2411820,0.2064969,0.2487768,0.2064969
2,CoType,mean(Collaboration) / mean(Service),0.1185947,1.1838889,0.017149565,69.03317,0,Inf,1.1502763,1.2175014,0.2111243,0.2499477,0.2111243
3,CoType,mean(Collaboration) / mean(Service),0.2371894,1.1635337,0.015798618,73.64782,0,Inf,1.1325690,1.1944985,0.2158271,0.2511221,0.2158271
4,CoType,mean(Collaboration) / mean(Service),0.3557841,1.1436721,0.014540888,78.65215,0,Inf,1.1151724,1.1721717,0.2206055,0.2523003,0.2206055
5,CoType,mean(Collaboration) / mean(Service),0.4743789,1.1242928,0.013382941,84.00939,0,Inf,1.0980627,1.1505228,0.2254592,0.2534821,0.2254592
6,CoType,mean(Collaboration) / mean(Service),0.5929736,1.1053851,0.012333043,89.62793,0,Inf,1.0812128,1.1295574,0.2303881,0.2546675,0.2303881
7,CoType,mean(Collaboration) / mean(Service),0.7115683,1.0869385,0.011401212,95.33535,0,Inf,1.0645926,1.1092845,0.2353920,0.2558566,0.2353920
8,CoType,mean(Collaboration) / mean(Service),0.8301630,1.0689428,0.010598889,100.85423,0,Inf,1.0481693,1.0897162,0.2404707,0.2570494,0.2404707
9,CoType,mean(Collaboration) / mean(Service),0.9487577,1.0513878,0.009938015,105.79454,0,Inf,1.0319096,1.0708659,0.2456237,0.2582457,0.2456237


In [40]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4c_comp_pre_co_lead.csv")
write.csv(res_pre_facpub, fname, row.names = FALSE)

In [ ]:
 [46] "lnex_ld_avg_before_year_prod_fac"            
 [47] "lnex_ld_avg_before_year_with_ih"             
 [48] "ex_ld_ratio_before_year_with_ih_bin"         
 [49] "ex_ld_max_before_year_with_ih_bin"           
[50] "lnex_ld_avg_before_year_participation"       
 [51] "ex_ld_ratio_before_year_participation_bin"   
 [52] "ex_ld_max_before_year_participation_bin"     
 [53] "lnex_ld_avg_before_year_co_lead"             
 [54] "ex_ld_ratio_before_year_co_lead_bin"         
 [55] "ex_ld_max_before_year_co_lead_bin"  

moderating <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_with_ih_bin"
moderating2 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_participation_bin"
moderating3 <- "lnex_ld_avg_before_year_prod_fac + ex_ld_max_before_year_co_lead_bin"

In [25]:
# 生成新变量名为 binary_ex_ld
data <- data %>%
  mutate(before_year_fac= if_else(ex_ld_avg_before_year_prod_fac == 0, "False", "True"))
data$before_year_fac <- factor(data$before_year_fac)
data <- within(data, before_year_fac <- relevel(before_year_fac, ref = 'False'))

In [31]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", disciplines, " | PublishedYear")
)
model_pre_facpub <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$before_year_fac=='False'), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_facpub)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 7,969
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                               Estimate Std. Error    z value
CoTypeCollaboration                            0.126156   0.059444   2.122262
CoTypeParticipation                            0.133038   0.104896   1.268285
lnnum_author                                   0.030602   0.047929   0.638490
internationalinternational                    -0.136724   0.060002  -2.278676
lnnum_reference                                0.128095   0.044759   2.861877
num_fac                                        0.161398   0.075014   2.151575
SDGTrue                                        0.052675   0.049469   1.064826
lnmean_career_age                             -0.033195   0.062703  -0.529410
lnex_ld_avg_avgimpact                         -0.138302   0.033108  -4.177278
lnex_ld_avg_insthindex                        -0.025546   0.03

In [65]:
# 1. 找到你这个连续变量的实际最小值和最大值（假设是 0 和 10，你需要改成你的实际极值）
min_val <- min(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
max_val <- max(data$lnex_ld_avg_before_year_prod_fac, na.rm=TRUE)
# 2. 运行估计
res_pre_facpub <- avg_comparisons(
    model_pre_facpub,
    variables = "CoType",
    comparison = "ratio",
    newdata = datagrid(
    model = model_pre_facpub,
    lnex_ld_avg_before_year_prod_fac = seq(min_val, max_val, length.out = 50) # 这里的 10 可以改成任意你想要的数字
  ),
    by = "lnex_ld_avg_before_year_prod_fac"
)
res_pre_facpub

rowid,term,contrast,lnex_ld_avg_before_year_prod_fac,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,CoType,mean(Collaboration) / mean(Service),0.0000000,1.058109,0.019753175,53.56652,0,Inf,1.019393,1.096824,0.2637936,0.2791224,0.2637936
2,CoType,mean(Collaboration) / mean(Service),0.1359301,1.058889,0.018873402,56.10480,0,Inf,1.021897,1.095880,0.2628715,0.2783516,0.2628715
3,CoType,mean(Collaboration) / mean(Service),0.2718602,1.059670,0.018003832,58.85804,0,Inf,1.024383,1.094957,0.2619515,0.2775822,0.2619515
4,CoType,mean(Collaboration) / mean(Service),0.4077903,1.060454,0.017146734,61.84584,0,Inf,1.026847,1.094061,0.2610335,0.2768141,0.2610335
5,CoType,mean(Collaboration) / mean(Service),0.5437204,1.061240,0.016304778,65.08768,0,Inf,1.029283,1.093197,0.2601176,0.2760473,0.2601176
6,CoType,mean(Collaboration) / mean(Service),0.6796505,1.062028,0.015481208,68.60112,0,Inf,1.031686,1.092371,0.2592038,0.2752818,0.2592038
7,CoType,mean(Collaboration) / mean(Service),0.8155806,1.062818,0.014679872,72.39969,0,Inf,1.034046,1.091590,0.2582921,0.2745176,0.2582921
8,CoType,mean(Collaboration) / mean(Service),0.9515107,1.063610,0.013905532,76.48830,0,Inf,1.036356,1.090865,0.2573825,0.2737547,0.2573825
9,CoType,mean(Collaboration) / mean(Service),1.0874408,1.064405,0.013163790,80.85853,0,Inf,1.038604,1.090205,0.2564750,0.2729932,0.2564750


In [66]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_facpub.csv")
write.csv(res_pre_facpub, fname, row.names = FALSE)

In [67]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_facpub,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                        & novel\_uzzi\_bin\\    
   Model:                                                                     & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                        & 0.077$^{***}$\\   
                                                                              & (0.025)\\   
   CoTypeParticipation                                                        & 0.183$^{***}$\\   
                                                                              & (0.036)\\   
   lnex\_ld\_avg\_before\_year\_prod\_fac                                     & -0.035$^{***}$\\   
                                                                              & (0.006)\\   
   lnnum\_author                                                              & -0.078$^{***}$\\   
                  

In [68]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_with_ih_bin  + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_pre_withih <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_withih)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                           Estimate Std. Error
CoTypeCollaboration                                        0.299043   0.031937
CoTypeParticipation                                        0.355810   0.055328
ex_ld_max_before_year_with_ih_binTrue                      0.357776   0.011822
lnnum_author                                              -0.081817   0.007394
internationalinternational                                -0.048462   0.009594
lnnum_reference                                            0.054297   0.008662
num_fac                                                    0.067452   0.006735
SDGTrue                                                    0.083733   0.008156
lnmean_career_age                                          0.006806   0.013098
lnex_ld_avg_avgimpact                             

In [69]:
# 每组 reg_class 的平均预测概率
res_pre_withih <- avg_comparisons(model_pre_withih, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_with_ih_bin')
res_pre_withih

term,contrast,ex_ld_max_before_year_with_ih_bin,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),False,1.1830643,0.022747259,52.00909,0.000000e+00,Inf,1.1384805,1.227648,0.2463877,0.3059910,0.2463877
CoType,mean(Collaboration) / mean(Service),True,1.0389235,0.007496342,138.59072,0.000000e+00,Inf,1.0242309,1.053616,0.2858751,0.2993944,0.2858751
CoType,mean(Participation) / mean(Service),False,1.2188867,0.037326152,32.65503,6.797902e-234,774.5661,1.1457288,1.292045,0.2463877,0.3181772,0.2463877
CoType,mean(Participation) / mean(Service),True,0.9831902,0.009145438,107.50609,0.000000e+00,Inf,0.9652655,1.001115,0.2858751,0.2800838,0.2858751


In [70]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_withih.csv")
write.csv(res_pre_withih, fname, row.names = FALSE)

In [71]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_withih,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\_bin\\    
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & 0.299$^{***}$\\   
                                                                                     & (0.032)\\   
   CoTypeParticipation                                                               & 0.356$^{***}$\\   
                                                                                     & (0.055)\\   
   ex\_ld\_max\_before\_year\_with\_ih\_binTrue                                      & 0.358$^{***}$\\   
                                                                                     & (0.012)\\   
   lnnum\_author                                               

In [72]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_participation_bin  + ", paper_level, "+", ex_controls, "+", moderating2, "+",disciplines, " | PublishedYear")
)
model_pre_partic <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_partic)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                                 Estimate
CoTypeCollaboration                                              0.188863
CoTypeParticipation                                              0.239312
ex_ld_max_before_year_participation_binTrue                      0.216772
lnnum_author                                                    -0.078782
internationalinternational                                      -0.045550
lnnum_reference                                                  0.057236
num_fac                                                          0.063163
SDGTrue                                                          0.085091
lnmean_career_age                                                0.018301
lnex_ld_avg_avgimpact                                           -0.272872
lnex_ld_avg_insthindex    

In [73]:
# 每组 reg_class 的平均预测概率
res_pre_partic <- avg_comparisons(model_pre_partic, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_participation_bin')
res_pre_partic

term,contrast,ex_ld_max_before_year_participation_bin,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),False,1.1123545,0.012681821,87.71252,0,Inf,1.0874986,1.1372104,0.2642670,0.3025795,0.2642670
CoType,mean(Collaboration) / mean(Service),True,1.0461863,0.008918249,117.30849,0,Inf,1.0287068,1.0636657,0.2875882,0.3033630,0.2875882
CoType,mean(Participation) / mean(Service),False,1.1430292,0.022816679,50.09621,0,Inf,1.0983093,1.1877490,0.2642670,0.3133303,0.2642670
CoType,mean(Participation) / mean(Service),True,0.9672328,0.009912748,97.57464,0,Inf,0.9478042,0.9866614,0.2875882,0.2765056,0.2875882


In [74]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_partic.csv")
write.csv(res_pre_partic, fname, row.names = FALSE)

In [75]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_partic,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                                   & novel\_uzzi\_bin\\    
   Model:                                                                                & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                                   & 0.189$^{***}$\\   
                                                                                         & (0.018)\\   
   CoTypeParticipation                                                                   & 0.239$^{***}$\\   
                                                                                         & (0.035)\\   
   lnnum\_author                                                                         & -0.079$^{***}$\\   
                                                                                         & (0.007)\\   
   internationalinternational 

In [76]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType*ex_ld_max_before_year_co_lead_bin  + ", paper_level, "+", ex_controls, "+", moderating3, "+",disciplines, " | PublishedYear")
)
model_pre_co_lead <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model_pre_co_lead)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                                           Estimate Std. Error
CoTypeCollaboration                                        0.267458   0.028977
CoTypeParticipation                                        0.269529   0.040795
ex_ld_max_before_year_co_lead_binTrue                      0.348245   0.011349
lnnum_author                                              -0.080558   0.007430
internationalinternational                                -0.047824   0.009599
lnnum_reference                                            0.053119   0.008662
num_fac                                                    0.066011   0.006742
SDGTrue                                                    0.083444   0.008156
lnmean_career_age                                          0.005323   0.013099
lnex_ld_avg_avgimpact                             

In [77]:
# 每组 reg_class 的平均预测概率
res_pre_co_lead <- avg_comparisons(model_pre_co_lead, variables = "CoType", comparison = 'ratio', by = 'ex_ld_max_before_year_co_lead_bin')
res_pre_co_lead

term,contrast,ex_ld_max_before_year_co_lead_bin,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high,predicted_lo,predicted_hi,predicted
<chr>,<chr>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CoType,mean(Collaboration) / mean(Service),False,1.163111,0.020427971,56.93719,0,Inf,1.1230732,1.2031494,0.2510753,0.3046128,0.2510753
CoType,mean(Collaboration) / mean(Service),True,1.033176,0.007494015,137.86681,0,Inf,1.0184879,1.0478639,0.2896318,0.3012384,0.2896318
CoType,mean(Participation) / mean(Service),False,1.164406,0.027393115,42.50726,0,Inf,1.1107168,1.2180958,0.2510753,0.3050516,0.2510753
CoType,mean(Participation) / mean(Service),True,0.977683,0.009356526,104.49209,0,Inf,0.9593446,0.9960215,0.2896318,0.2818850,0.2896318


In [78]:
fname = paste0(main_path, "GraduationPaper/RevisetoJournal/R_ex_ld_h4_comp_pre_co_lead.csv")
write.csv(res_pre_co_lead, fname, row.names = FALSE)

In [79]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_pre_co_lead,
                           keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                           se = "hetero",
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                                                               & novel\_uzzi\_bin\\    
   Model:                                                                            & (1)\\  
   \midrule
   \emph{Variables}\\
   CoTypeCollaboration                                                               & 0.267$^{***}$\\   
                                                                                     & (0.029)\\   
   CoTypeParticipation                                                               & 0.270$^{***}$\\   
                                                                                     & (0.041)\\   
   lnnum\_author                                                                     & -0.081$^{***}$\\   
                                                                                     & (0.007)\\   
   internationalinternational                                 

# 补充一个更deep的point，曾经开展过“Co-lead”,后续合作/参与的收益受损更严重

# Step-wised

In [81]:
fml1 <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", disciplines, " | PublishedYear")
)
fml2 <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+",disciplines, " | PublishedYear")
)
fml3 <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
fml4 <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_1 <- feglm(fml1, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_2 <- feglm(fml2, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_3 <- feglm(fml3, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_4 <- feglm(fml4, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")

In [82]:
model_1
model_2
model_3
model_4

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error  z value
text_fac_scientistStaffPart                   0.119764   0.009233 12.97138
Agricultural.and.Biological.Sciences          0.765513   0.023779 32.19238
Arts.and.Humanities                           1.861914   0.072157 25.80357
Biochemistry..Genetics.and.Molecular.Biology  0.513419   0.012218 42.02188
Business..Management.and.Accounting          -0.392942   0.087433 -4.49420
Chemical.Engineering                          0.160682   0.020746  7.74520
Chemistry                                     0.495371   0.010969 45.16053
Computer.Science                              0.941051   0.039255 23.97277
                                               Pr(>|z|)    
text_fac_scientistStaffPart                   < 2.2e-16 ***
Agricultural.and.Biological.Sciences         

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                      Estimate Std. Error    z value  Pr(>|z|)
text_fac_scientistStaffPart           0.140662   0.009737  14.446215 < 2.2e-16
lnnum_author                         -0.132290   0.007077 -18.692584 < 2.2e-16
internationalinternational           -0.011325   0.008701  -1.301585   0.19306
lnnum_reference                      -0.013291   0.008424  -1.577673   0.11464
num_fac                               0.055900   0.006562   8.519151 < 2.2e-16
SDGTrue                               0.077598   0.008111   9.567509 < 2.2e-16
lnmean_career_age                    -0.005025   0.012682  -0.396260   0.69191
Agricultural.and.Biological.Sciences  0.756770   0.023770  31.837440 < 2.2e-16
                                        
text_fac_scientistStaffPart          ***
lnnum_author                         ***
intern

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error   z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.104018   0.009877  10.53152  < 2.2e-16 ***
lnnum_author                -0.069970   0.007090  -9.86959  < 2.2e-16 ***
internationalinternational  -0.032494   0.009576  -3.39316 6.9092e-04 ***
lnnum_reference              0.054540   0.008631   6.31915 2.6300e-10 ***
num_fac                      0.064876   0.006621   9.79849  < 2.2e-16 ***
SDGTrue                      0.082338   0.008139  10.11637  < 2.2e-16 ***
lnmean_career_age            0.034551   0.012865   2.68557 7.2405e-03 ** 
lnex_ld_avg_avgimpact       -0.268328   0.007289 -36.81067  < 2.2e-16 ***
... 30 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -181,994.4   

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error    z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.064958   0.010007   6.491372 8.5058e-11 ***
lnnum_author                -0.087859   0.007299 -12.037309  < 2.2e-16 ***
internationalinternational  -0.046539   0.009593  -4.851415 1.2258e-06 ***
lnnum_reference              0.052699   0.008659   6.085706 1.1598e-09 ***
num_fac                      0.066565   0.006729   9.891698  < 2.2e-16 ***
SDGTrue                      0.084342   0.008152  10.345615  < 2.2e-16 ***
lnmean_career_age            0.011241   0.013083   0.859195 3.9023e-01    
lnex_ld_avg_avgimpact       -0.274514   0.007390 -37.145608  < 2.2e-16 ***
... 32 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -181

In [87]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model_1, model_2, model_3, model_4,
                    keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + pr2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)           & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    text\_fac\_scientistStaffPart                       & 0.120$^{***}$ & 0.141$^{***}$  & 0.104$^{***}$  & 0.065$^{***}$\\                                                           & (0.009)       & (0.010)        & (0.010)        & (0.010)\\       lnnum\_author                                       &               & -0.132$^{***}$ & -0.070$^{***}$ & -0.088$^{***}$\\                                                           &               & (0.007)        & (0.007)        & (0.007)\\       internationalinternational                          &               & -0.011         & -0.033$^{***}$ & -0.046$^{***}$\\                                                           &               & (0.009)    

In [91]:
fmlrd <- as.formula(
  paste0("novel_uzzi_bin ~ text_fac_scientist + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_ps_bin <- feglm(fmlrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
model_nps_bin <- feglm(fmlrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
model_hs_bin <- feglm(fmlrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
model_ls_bin <- feglm(fmlrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")

NOTE: 6 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 6 fixed-effects (8 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



In [93]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model_ps_bin, model_nps_bin, model_hs_bin, model_ls_bin,
                    keep = c("text_fac_scientist", paper_vars, ex_vars, moderating_var),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + pr2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    text\_fac\_scientistStaffPart                       & 0.059$^{***}$  & 0.318$^{***}$  & 0.388$^{***}$  & 0.131$^{***}$\\                                                           & (0.011)        & (0.036)        & (0.038)        & (0.022)\\       lnnum\_author                                       & -0.177$^{***}$ & 0.345$^{***}$  & 0.320$^{***}$  & 0.233$^{***}$\\                                                           & (0.008)        & (0.024)        & (0.026)        & (0.017)\\       internationalinternational                          & -0.075$^{***}$ & 0.108$^{***}$  & 0.059$^{**}$   & 0.091$^{***}$\\                                                           & (0.011)        & (0.024

In [92]:
model_ps_bin
model_nps_bin
model_hs_bin
model_ls_bin

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,575
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error   z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.058672   0.010578   5.54659 2.9130e-08 ***
lnnum_author                -0.176638   0.008240 -21.43792  < 2.2e-16 ***
internationalinternational  -0.075425   0.010529  -7.16378 7.8480e-13 ***
lnnum_reference              0.077542   0.009242   8.39026  < 2.2e-16 ***
num_fac                      0.102222   0.007275  14.05173  < 2.2e-16 ***
SDGTrue                      0.032981   0.008921   3.69687 2.1827e-04 ***
lnmean_career_age           -0.050889   0.014203  -3.58311 3.3953e-04 ***
lnex_ld_avg_avgimpact       -0.223156   0.008253 -27.03788  < 2.2e-16 ***
... 32 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -153,671.4   

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,294
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error   z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.317739   0.036403   8.72833  < 2.2e-16 ***
lnnum_author                 0.344890   0.023766  14.51212  < 2.2e-16 ***
internationalinternational   0.107954   0.024142   4.47167 7.7613e-06 ***
lnnum_reference             -0.088392   0.026780  -3.30065 9.6461e-04 ***
num_fac                     -0.131859   0.018676  -7.06033 1.6610e-12 ***
SDGTrue                      0.260102   0.021372  12.16997  < 2.2e-16 ***
lnmean_career_age            0.266285   0.035370   7.52850 5.1325e-14 ***
lnex_ld_avg_avgimpact       -0.534767   0.021066 -25.38509  < 2.2e-16 ***
... 21 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -26,058.1   Ad

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,409
Fixed-effects: PublishedYear: 40
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error   z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.388395   0.038336  10.13135  < 2.2e-16 ***
lnnum_author                 0.319584   0.026505  12.05740  < 2.2e-16 ***
internationalinternational   0.058681   0.026496   2.21472 2.6779e-02 *  
lnnum_reference              0.110130   0.027640   3.98449 6.7624e-05 ***
num_fac                     -0.123160   0.020739  -5.93845 2.8772e-09 ***
SDGTrue                      0.224109   0.023466   9.55045  < 2.2e-16 ***
lnmean_career_age            0.184938   0.038681   4.78107 1.7436e-06 ***
lnex_ld_avg_avgimpact       -0.334968   0.022019 -15.21267  < 2.2e-16 ***
... 32 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -21,581.4   Ad

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,590
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                             Estimate Std. Error   z value   Pr(>|z|)    
text_fac_scientistStaffPart  0.131165   0.022359   5.86637 4.4544e-09 ***
lnnum_author                 0.232568   0.016875  13.78196  < 2.2e-16 ***
internationalinternational   0.091356   0.017481   5.22596 1.7325e-07 ***
lnnum_reference              0.100861   0.017830   5.65668 1.5433e-08 ***
num_fac                     -0.074282   0.012579  -5.90498 3.5268e-09 ***
SDGTrue                      0.157392   0.015172  10.37373  < 2.2e-16 ***
lnmean_career_age            0.149261   0.024350   6.12989 8.7939e-10 ***
lnex_ld_avg_avgimpact       -0.386505   0.014080 -27.45105  < 2.2e-16 ***
... 31 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -50,616.2   Ad

In [88]:
fmlt1 <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", disciplines, " | PublishedYear")
)
fmlt2 <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+",disciplines, " | PublishedYear")
)
fmlt3 <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
fmlt4 <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model_t1 <- feglm(fmlt1, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_t2 <- feglm(fmlt2, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_t3 <- feglm(fmlt3, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
model_t4 <- feglm(fmlt4, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")

In [89]:
model_t1
model_t2
model_t3
model_t4

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error  z value
CoTypeCollaboration                           0.159156   0.010644 14.95214
CoTypeParticipation                           0.037926   0.014258  2.65998
Agricultural.and.Biological.Sciences          0.765525   0.023788 32.18123
Arts.and.Humanities                           1.862658   0.072192 25.80129
Biochemistry..Genetics.and.Molecular.Biology  0.510660   0.012224 41.77467
Business..Management.and.Accounting          -0.395923   0.087547 -4.52242
Chemical.Engineering                          0.160474   0.020743  7.73635
Chemistry                                     0.494542   0.010971 45.07566
                                               Pr(>|z|)    
CoTypeCollaboration                           < 2.2e-16 ***
CoTypeParticipation                          

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error    z value   Pr(>|z|)    
CoTypeCollaboration         0.166329   0.011004  15.115250  < 2.2e-16 ***
CoTypeParticipation         0.083954   0.014865   5.647811 1.6250e-08 ***
lnnum_author               -0.127904   0.007167 -17.846845  < 2.2e-16 ***
internationalinternational -0.011376   0.008700  -1.307567 1.9102e-01    
lnnum_reference            -0.012900   0.008424  -1.531293 1.2570e-01    
num_fac                     0.055591   0.006563   8.470389  < 2.2e-16 ***
SDGTrue                     0.077079   0.008112   9.501670  < 2.2e-16 ***
lnmean_career_age          -0.007455   0.012693  -0.587354 5.5697e-01    
... 26 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -182,977.3   

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error  z value   Pr(>|z|)    
CoTypeCollaboration         0.134576   0.011132 12.08924  < 2.2e-16 ***
CoTypeParticipation         0.036306   0.015001  2.42016 1.5514e-02 *  
lnnum_author               -0.064427   0.007187 -8.96456  < 2.2e-16 ***
internationalinternational -0.032915   0.009577 -3.43686 5.8849e-04 ***
lnnum_reference             0.055179   0.008632  6.39273 1.6295e-10 ***
num_fac                     0.064480   0.006623  9.73558  < 2.2e-16 ***
SDGTrue                     0.081746   0.008141 10.04169  < 2.2e-16 ***
lnmean_career_age           0.031941   0.012874  2.48095 1.3103e-02 *  
... 31 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -181,977.3   Adj. Pseudo R2: 0.

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error    z value   Pr(>|z|)    
CoTypeCollaboration         0.093664   0.011280   8.303659  < 2.2e-16 ***
CoTypeParticipation         0.001874   0.015079   0.124291 9.0108e-01    
lnnum_author               -0.082752   0.007391 -11.196746  < 2.2e-16 ***
internationalinternational -0.046879   0.009593  -4.886663 1.0256e-06 ***
lnnum_reference             0.053317   0.008660   6.156694 7.4279e-10 ***
num_fac                     0.066123   0.006732   9.822587  < 2.2e-16 ***
SDGTrue                     0.083797   0.008154  10.276936  < 2.2e-16 ***
lnmean_career_age           0.008714   0.013093   0.665580 5.0568e-01    
... 33 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -181,536.1   

In [90]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model_t1, model_t2, model_t3, model_t4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + pr2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)           & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.159$^{***}$ & 0.166$^{***}$  & 0.135$^{***}$  & 0.094$^{***}$\\                                                           & (0.011)       & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.038$^{***}$ & 0.084$^{***}$  & 0.036$^{**}$   & 0.002\\                                                           & (0.014)       & (0.015)        & (0.015)        & (0.015)\\       lnnum\_author                                       &               & -0.128$^{***}$ & -0.064$^{***}$ & -0.083$^{***}$\\                                                           &               & (0.007)        & (0.

In [94]:
fmltrd <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
tmodel_ps_bin <- feglm(fmltrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
tmodel_nps_bin <- feglm(fmltrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Physical.Sciences==0), ], family = binomial("logit"), vcov = "hetero")
tmodel_hs_bin <- feglm(fmltrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Health.Sciences==1), ], family = binomial("logit"), vcov = "hetero")
tmodel_ls_bin <- feglm(fmltrd, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0)&(data$domain_Life.Sciences==1), ], family = binomial("logit"), vcov = "hetero")

NOTE: 6 fixed-effects (10 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 6 fixed-effects (8 observations) removed because of only 0 (or only 1) outcomes.

NOTE: 1 fixed-effect (1 observation) removed because of only 0 (or only 1) outcomes.



In [95]:
tmodel_ps_bin
tmodel_nps_bin
tmodel_hs_bin
tmodel_ls_bin

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 253,575
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error    z value   Pr(>|z|)    
CoTypeCollaboration         0.081750   0.011854   6.896493 5.3302e-12 ***
CoTypeParticipation         0.006896   0.015894   0.433865 6.6439e-01    
lnnum_author               -0.172038   0.008338 -20.633222  < 2.2e-16 ***
internationalinternational -0.075657   0.010529  -7.185651 6.6887e-13 ***
lnnum_reference             0.078010   0.009242   8.440694  < 2.2e-16 ***
num_fac                     0.101840   0.007277  13.994286  < 2.2e-16 ***
SDGTrue                     0.032604   0.008922   3.654131 2.5805e-04 ***
lnmean_career_age          -0.052982   0.014213  -3.727710 1.9323e-04 ***
... 33 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -153,662.3   

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 41,294
Fixed-effects: PublishedYear: 38
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error  z value   Pr(>|z|)    
CoTypeCollaboration         0.281504   0.042795  6.57803 4.7673e-11 ***
CoTypeParticipation         0.395612   0.059091  6.69494 2.1576e-11 ***
lnnum_author                0.344441   0.023759 14.49700  < 2.2e-16 ***
internationalinternational  0.107923   0.024142  4.47031 7.8106e-06 ***
lnnum_reference            -0.088769   0.026784 -3.31421 9.1903e-04 ***
num_fac                    -0.131631   0.018687 -7.04403 1.8676e-12 ***
SDGTrue                     0.260535   0.021376 12.18810  < 2.2e-16 ***
lnmean_career_age           0.268377   0.035399  7.58155 3.4146e-14 ***
... 22 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -26,056.8   Adj. Pseudo R2: 0.07

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 33,409
Fixed-effects: PublishedYear: 40
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error  z value   Pr(>|z|)    
CoTypeCollaboration         0.347392   0.044522  7.80271 6.0591e-15 ***
CoTypeParticipation         0.478942   0.063318  7.56410 3.9056e-14 ***
lnnum_author                0.318061   0.026497 12.00377  < 2.2e-16 ***
internationalinternational  0.058039   0.026499  2.19023 2.8507e-02 *  
lnnum_reference             0.110106   0.027655  3.98134 6.8527e-05 ***
num_fac                    -0.123250   0.020755 -5.93839 2.8783e-09 ***
SDGTrue                     0.224452   0.023469  9.56391  < 2.2e-16 ***
lnmean_career_age           0.187322   0.038706  4.83968 1.3005e-06 ***
... 33 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -21,579.7   Adj. Pseudo R2: 0.05

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 77,590
Fixed-effects: PublishedYear: 48
Standard-errors: Heteroskedasticity-robust 
                            Estimate Std. Error  z value   Pr(>|z|)    
CoTypeCollaboration         0.112862   0.025274  4.46548 7.9888e-06 ***
CoTypeParticipation         0.177887   0.037424  4.75325 2.0017e-06 ***
lnnum_author                0.232349   0.016872 13.77098  < 2.2e-16 ***
internationalinternational  0.091095   0.017482  5.21065 1.8818e-07 ***
lnnum_reference             0.100646   0.017834  5.64361 1.6652e-08 ***
num_fac                    -0.074019   0.012581 -5.88365 4.0132e-09 ***
SDGTrue                     0.157520   0.015173 10.38170  < 2.2e-16 ***
lnmean_career_age           0.150727   0.024366  6.18588 6.1757e-10 ***
... 32 coefficients remaining (display them with summary() or use argument n)
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -50,614.9   Adj. Pseudo R2: 0.05

In [96]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(tmodel_ps_bin, tmodel_nps_bin, tmodel_hs_bin, tmodel_ls_bin,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + pr2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.082$^{***}$  & 0.282$^{***}$  & 0.347$^{***}$  & 0.113$^{***}$\\                                                           & (0.012)        & (0.043)        & (0.044)        & (0.025)\\       CoTypeParticipation                                 & 0.007          & 0.396$^{***}$  & 0.479$^{***}$  & 0.178$^{***}$\\                                                           & (0.016)        & (0.059)        & (0.063)        & (0.037)\\       lnnum\_author                                       & -0.172$^{***}$ & 0.344$^{***}$  & 0.318$^{***}$  & 0.232$^{***}$\\                                                           & (0.008)        & (0.024

In [90]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+",disciplines, " | PublishedYear")
)
model3 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model3)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.143687   0.011097  12.948869
CoTypeParticipation                           0.047011   0.014969   3.140640
lnnum_author                                 -0.068658   0.007116  -9.648419
internationalinternational                   -0.031089   0.009559  -3.252299
lnnum_reference                               0.061520   0.008603   7.150632
num_fac                                       0.059498   0.006605   9.007958
SDGTrue                                       0.092371   0.008114  11.384687
lnmean_career_age                             0.033050   0.012852   2.571617
lnex_ld_avg_avgimpact                        -0.264828   0.007259 -36.483771
lnex_ld_avg_insthindex                       -0.049529   0.007468  -6.

In [91]:
fml <- as.formula(
  paste0("novel_uzzi_bin ~ CoType + ", paper_level, "+", ex_controls, "+", moderating, "+",disciplines, " | PublishedYear")
)
model4 <- feglm(fml, data = data[(data$paper_type =='article')&(data$knowledge_proximity_mean>0), ], family = binomial("logit"), vcov = "hetero")
summary(model4)

GLM estimation, family = binomial, Dep. Var.: novel_uzzi_bin
Observations: 294,879
Fixed-effects: PublishedYear: 51
Standard-errors: Heteroskedasticity-robust 
                                              Estimate Std. Error    z value
CoTypeCollaboration                           0.099926   0.011245   8.886446
CoTypeParticipation                           0.010299   0.015044   0.684582
lnnum_author                                 -0.087020   0.007328 -11.874570
internationalinternational                   -0.045118   0.009576  -4.711696
lnnum_reference                               0.059057   0.008633   6.841129
num_fac                                       0.062924   0.006715   9.369983
SDGTrue                                       0.093974   0.008128  11.561174
lnmean_career_age                             0.012005   0.013072   0.918354
lnex_ld_avg_avgimpact                        -0.269872   0.007355 -36.690437
lnex_ld_avg_insthindex                       -0.051397   0.007510  -6.

In [92]:
# 直接把 4 个模型并排放在一起
tab_latex <- etable(model1, model2, model3, model4,
                    keep = c("CoType", paper_vars, ex_vars, moderating_var, disciplines_vars),
                    se = "hetero",
                    tex = TRUE,
                    digits = 3,
                    fitstat = ~ n + r2 + ar2) # 可选：指定要在底部报告的统计量（如样本量、R方、调整R方）

# 打印出可以直接复制到 LaTeX 的代码
cat(tab_latex)

\begingroup \centering \begin{tabular}{lcccc}    \tabularnewline \midrule \midrule    Dependent Variable: & \multicolumn{4}{c}{novel\_uzzi\_bin}\\    Model:                                              & (1)            & (2)            & (3)            & (4)\\      \midrule    \emph{Variables}\\    CoTypeCollaboration                                 & 0.168$^{***}$  & 0.176$^{***}$  & 0.144$^{***}$  & 0.100$^{***}$\\                                                           & (0.011)        & (0.011)        & (0.011)        & (0.011)\\       CoTypeParticipation                                 & 0.047$^{***}$  & 0.095$^{***}$  & 0.047$^{***}$  & 0.010\\                                                           & (0.014)        & (0.015)        & (0.015)        & (0.015)\\       Arts.and.Humanities                                 & 1.84$^{***}$   & 1.81$^{***}$   & 1.76$^{***}$   & 1.74$^{***}$\\                                                           & (0.072)        & (0.072)        